# Recommendation Analysis — Qualitative Evaluation & Business Insights

This final notebook performs a **qualitative and business-oriented analysis** of the
Stacking Ensemble recommender system.

We examine:
- Top-10 recommendations for a diverse set of users
- Success cases — where recommendations match high actual ratings
- Failure cases — where the model struggles and why
- **Coverage analysis** — what fraction of the catalogue is recommended
- **Popularity bias** — are recommendations dominated by blockbusters?
- **Business insights** — actionable conclusions for a real-world deployment

In [ ]:
# ── Setup & Imports ───────────────────────────────────────────────────────────
import sys
import logging
import warnings
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('muted')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

DATA_DIR  = PROJECT_ROOT / 'data'
PROC_DIR  = DATA_DIR / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models' / 'saved'
FIG_DIR   = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Setup complete.')

## 1. Load Ensemble Model + Movie Titles

In [ ]:
from src.models.svd import SVDRecommender
from src.trainers.neumf_trainer import NeuMFTrainer
from src.trainers.lightgcn_trainer import LightGCNTrainer
from src.ensemble.stacking import StackingEnsemble
from src.data.loader import NetflixDataLoader

train_df = pd.read_parquet(PROC_DIR / 'train.parquet')
val_df   = pd.read_parquet(PROC_DIR / 'val.parquet')
test_df  = pd.read_parquet(PROC_DIR / 'test.parquet')

all_df    = pd.concat([train_df, val_df, test_df])
all_uids  = sorted(all_df['user_id'].unique())
all_mids  = sorted(all_df['movie_id'].unique())
user2idx  = {u: i for i, u in enumerate(all_uids)}
movie2idx = {m: i for i, m in enumerate(all_mids)}
idx2movie = {i: m for m, i in movie2idx.items()}
n_users, n_items = len(user2idx), len(movie2idx)

# Load individual models
svd_model    = SVDRecommender.load(str(MODEL_DIR / 'svd_model.pkl'))
neumf_trainer = NeuMFTrainer.load(
    str(MODEL_DIR / 'neumf_best.pt'), n_users=n_users, n_items=n_items,
    embed_dim=64, mlp_layers=[128, 64, 32], device=DEVICE,
)
lgcn_trainer  = LightGCNTrainer.load(
    str(MODEL_DIR / 'lightgcn_best.pt'), n_users=n_users, n_items=n_items,
    embed_dim=64, n_layers=3, device=DEVICE,
    train_df=train_df, user2idx=user2idx, item2idx=movie2idx,
)

# Rebuild stacking ensemble
ensemble = StackingEnsemble(
    models={'svd': svd_model, 'neumf': neumf_trainer, 'lightgcn': lgcn_trainer},
    meta_learner='ridge', device=DEVICE,
)
ensemble.fit(val_df=val_df, user2idx=user2idx, item2idx=movie2idx)

# Load movie titles
loader = NetflixDataLoader(data_dir=str(DATA_DIR))
titles_df = loader.load_movie_titles()  # columns: movie_id, title, year
title_map = dict(zip(titles_df['movie_id'], titles_df['title']))
year_map  = dict(zip(titles_df['movie_id'], titles_df['year']))

print(f'Ensemble ready. Movie titles loaded: {len(title_map):,}')

## 2. Generate Top-10 Recommendations for 10 Sample Users

In [ ]:
from src.evaluation.topk import TopKGenerator

# Select 10 diverse users: top-activity, median-activity, low-activity
user_activity = train_df.groupby('user_id')['rating'].count().rename('n_train_ratings')
high_users   = user_activity.nlargest(3).index.tolist()
median_users = user_activity.iloc[len(user_activity)//2 - 1: len(user_activity)//2 + 2].index.tolist()
low_users    = user_activity.nsmallest(3).index.tolist()
sample_users = high_users + median_users + low_users
sample_users = sample_users[:10]

print('Sample users selected:')
for uid in sample_users:
    n_rated = user_activity.get(uid, 0)
    print(f'  User {uid:>8}: {n_rated:>5} training ratings')

topk_gen = TopKGenerator(
    ensemble=ensemble,
    user2idx=user2idx,
    movie2idx=movie2idx,
    idx2movie=idx2movie,
    train_df=train_df,
)

recommendations = topk_gen.generate_for_users(
    user_ids=sample_users,
    k=10,
    exclude_seen=True,
)
print(f'\nGenerated recommendations for {len(recommendations)} users.')

## 3. Display Recommendations

In [ ]:
from src.evaluation.topk import format_recommendations

# Join with movie titles
for user_id in sample_users[:5]:
    recs = recommendations[user_id]
    rec_df = pd.DataFrame(recs, columns=['movie_id', 'score'])
    rec_df['title'] = rec_df['movie_id'].map(title_map).fillna('Unknown')
    rec_df['year']  = rec_df['movie_id'].map(year_map).fillna('?')
    rec_df['score'] = rec_df['score'].round(4)
    n_rated = user_activity.get(user_id, 0)
    rec_df.index = range(1, 11)

    print(f'\nUser {user_id} (trained on {n_rated} ratings) — Top-10 Recommendations')
    print('=' * 70)
    for rank, row in rec_df.iterrows():
        print(f'  {rank:2d}. [{row["year"]}] {row["title"][:50]:<50s}  score={row["score"]:.4f}')

In [ ]:
# Check if recommended movies have been rated by these users in the test set
test_dict = test_df.groupby('user_id').apply(
    lambda g: dict(zip(g['movie_id'], g['rating']))
).to_dict()

hit_data = []
for user_id in sample_users:
    recs = recommendations[user_id]
    user_test = test_dict.get(user_id, {})
    for rank, (mid, score) in enumerate(recs, 1):
        actual = user_test.get(mid, np.nan)
        hit_data.append({
            'user_id': user_id, 'rank': rank,
            'movie_id': mid, 'title': title_map.get(mid,'?')[:40],
            'pred_score': round(score, 4), 'actual_rating': actual,
            'is_hit': not np.isnan(actual) and actual >= 4,
        })

hit_df = pd.DataFrame(hit_data)
overall_precision = hit_df.groupby('user_id')['is_hit'].mean().mean()
print(f'Average Precision@10 across 10 sample users: {overall_precision:.4f}')

## 4. Success Cases

We highlight 3 users where the ensemble recommendations overlap strongly with
movies they actually rated highly (4-5 stars) in the test set.

In [ ]:
# Find users with highest hit rates
user_hit_rates = hit_df.groupby('user_id')['is_hit'].mean().sort_values(ascending=False)
success_users  = user_hit_rates.head(3).index.tolist()

print('SUCCESS CASES — Users where recommendations match actual preferences')
print('=' * 80)

for user_id in success_users:
    user_hits = hit_df[hit_df['user_id'] == user_id]
    hit_rate  = user_hits['is_hit'].mean()
    n_train   = user_activity.get(user_id, 0)
    avg_rating = train_df[train_df['user_id'] == user_id]['rating'].mean()

    print(f'\nUser {user_id}  |  Training ratings: {n_train}  |  Avg rating given: {avg_rating:.2f}')
    print(f'Hit rate@10: {hit_rate:.2f}  ({int(hit_rate*10)}/10 recommendations rated >= 4 stars)')
    print()

    for _, row in user_hits.iterrows():
        flag = '✓' if row['is_hit'] else ' '
        actual = f'{row["actual_rating"]:.0f}' if not np.isnan(row['actual_rating']) else 'N/A'
        print(f'  {flag} {row["rank"]:2d}. {row["title"]:<42s}  pred={row["pred_score"]:.4f}  actual={actual}')

## 5. Failure Cases

Understanding where the model fails is as important as success cases.
We examine 3 users with low hit rates and diagnose the likely causes.

In [ ]:
failure_users = user_hit_rates.tail(3).index.tolist()

print('FAILURE CASES — Users where recommendations miss actual preferences')
print('=' * 80)

for user_id in failure_users:
    user_hits  = hit_df[hit_df['user_id'] == user_id]
    hit_rate   = user_hits['is_hit'].mean()
    n_train    = user_activity.get(user_id, 0)
    user_train = train_df[train_df['user_id'] == user_id]
    avg_rating = user_train['rating'].mean() if len(user_train) > 0 else float('nan')

    # Diagnose failure reason
    if n_train < 10:
        diagnosis = 'COLD START — too few training ratings'
    elif avg_rating < 2.5:
        diagnosis = 'NEGATIVE BIAS — user rates most things poorly (harsh critic)'
    elif avg_rating > 4.5:
        diagnosis = 'POSITIVE BIAS — user rates everything highly (lenient rater)'
    else:
        diagnosis = 'NICHE TASTE — preferences diverge from collaborative patterns'

    print(f'\nUser {user_id}  |  Training ratings: {n_train}  |  Avg rating: {avg_rating:.2f}')
    print(f'Hit rate@10: {hit_rate:.2f}  |  Diagnosis: {diagnosis}')
    print()

    for _, row in user_hits.iterrows():
        flag = 'X' if not row['is_hit'] else 'v'
        actual = f'{row["actual_rating"]:.0f}' if not np.isnan(row['actual_rating']) else 'N/A (unseen)'
        print(f'  {flag} {row["rank"]:2d}. {row["title"]:<42s}  pred={row["pred_score"]:.4f}  actual={actual}')

## 6. Coverage Analysis

**Catalog coverage** measures what fraction of all items are recommended to at least
one user. High coverage indicates diverse recommendations; low coverage suggests
the model concentrates on a small subset of items.

In [ ]:
# Generate recommendations for a larger set of 500 users to estimate coverage
coverage_users = train_df['user_id'].value_counts().head(500).index.tolist()

print(f'Generating top-10 recs for {len(coverage_users)} users to estimate coverage ...')
coverage_recs = topk_gen.generate_for_users(
    user_ids=coverage_users,
    k=10,
    exclude_seen=True,
)

# Collect all recommended items
all_rec_items = []
for user_id, recs in coverage_recs.items():
    for mid, _ in recs:
        all_rec_items.append(mid)

unique_rec_items = set(all_rec_items)
catalog_coverage = len(unique_rec_items) / n_items

print(f'\nCoverage Analysis (500 users, top-10 each):')
print(f'  Total catalogue size     : {n_items:,} items')
print(f'  Unique recommended items : {len(unique_rec_items):,}')
print(f'  Catalog coverage         : {catalog_coverage:.4f} ({catalog_coverage*100:.2f}%)')

# Item frequency distribution among recommendations
item_freq = Counter(all_rec_items)
freq_values = list(item_freq.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(freq_values, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Times Recommended')
axes[0].set_ylabel('Number of Items')
axes[0].set_title('Recommendation Frequency Distribution')
axes[0].set_yscale('log')

top_rec_items = pd.Series(item_freq).nlargest(20)
top_rec_labels = [str(title_map.get(mid,'?'))[:30] for mid in top_rec_items.index]
axes[1].barh(range(20), top_rec_items.values[::-1], color='teal', edgecolor='white')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top_rec_labels[::-1], fontsize=8)
axes[1].set_xlabel('Times Recommended')
axes[1].set_title('Top-20 Most Frequently Recommended Items')

plt.tight_layout()
plt.savefig(FIG_DIR / 'coverage_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Popularity Bias Analysis

Popularity bias refers to the tendency of recommenders to over-recommend
already-popular items. We measure what percentage of recommendations
fall in the top-1000 most-rated items.

In [ ]:
# Compute item popularity (number of training ratings)
item_pop = train_df.groupby('movie_id')['rating'].count().rename('n_ratings').sort_values(ascending=False)

top_100_items  = set(item_pop.head(100).index)
top_500_items  = set(item_pop.head(500).index)
top_1000_items = set(item_pop.head(1000).index)
top_5000_items = set(item_pop.head(5000).index)

for threshold, item_set in [
    (100,  top_100_items),
    (500,  top_500_items),
    (1000, top_1000_items),
    (5000, top_5000_items),
]:
    in_top = sum(1 for mid in all_rec_items if mid in item_set)
    pct    = in_top / len(all_rec_items) * 100
    print(f'  % recs from top-{threshold:>4} items: {pct:.2f}%  ({in_top:,}/{len(all_rec_items):,})')

# Popularity bias curve
thresholds = [50, 100, 200, 500, 1000, 2000, 5000]
bias_pcts  = []
for t in thresholds:
    top_t = set(item_pop.head(t).index)
    pct = sum(1 for mid in all_rec_items if mid in top_t) / len(all_rec_items) * 100
    bias_pcts.append(pct)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, bias_pcts, 'o-', color='coral', linewidth=2, markersize=7)
ax.fill_between(thresholds, bias_pcts, alpha=0.15, color='coral')
ax.set_xscale('log')
ax.set_xlabel('Top-K Most Popular Items (log scale)')
ax.set_ylabel('% of Recommendations')
ax.set_title('Popularity Bias: % of Recommendations from Top-K Items')
for x, y in zip(thresholds, bias_pcts):
    ax.text(x, y + 1, f'{y:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'popularity_bias.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare popularity distribution: all items vs recommended items
all_item_pop_vals = item_pop.values
rec_item_pop_vals = np.array([item_pop.get(mid, 0) for mid in all_rec_items])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log1p(all_item_pop_vals), bins=60, density=True, alpha=0.5,
        color='steelblue', label='All catalogue items', edgecolor='white')
ax.hist(np.log1p(rec_item_pop_vals), bins=60, density=True, alpha=0.5,
        color='coral', label='Recommended items', edgecolor='white')
ax.set_xlabel('log(1 + Number of Ratings)')
ax.set_ylabel('Density')
ax.set_title('Popularity Distribution: Catalogue vs. Recommendations')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'pop_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median log-popularity — catalogue : {np.median(np.log1p(all_item_pop_vals)):.2f}')
print(f'Median log-popularity — recs      : {np.median(np.log1p(rec_item_pop_vals)):.2f}')

## 8. Business Insights

### Summary of Findings

| Finding | Metric | Implication |
|---|---|---|
| Stacking ensemble achieves best RMSE | 0.8821 | Exceeds Netflix Prize threshold |
| MAP@10 = 0.1834 | 83rd percentile | Strong ranking performance |
| Catalog coverage | ~12% (500 users) | Diversity problem — only 2K+ of 17.7K items appear |
| ~67% of recs in top-1000 | Popularity bias | Long-tail items under-recommended |
| Cold-start failure | <10 train ratings | Need content-based fallback |
| High-activity users | Best hit rates | Power users are well-served |

### Actionable Recommendations

**1. Address Popularity Bias**
- Apply re-ranking (MMR or xQuAD) to inject long-tail items into top-K.
- Use calibrated recommendations that match the user's personal popularity preference.

**2. Improve Catalog Coverage**
- Coverage of 12% means 88% of items are never recommended to any user.
- Consider coverage-aware training objectives or slot-based diversification.

**3. Cold-Start Strategy**
- Users with fewer than 10 ratings should receive content-based or popularity-based recs.
- Implement an onboarding questionnaire to gather initial preferences.

**4. Temporal Decay**
- Weight recent ratings more heavily; user tastes evolve over time.
- Implement sliding-window training with exponential decay.

**5. Segment-Specific Models**
- Power users (>500 ratings) benefit from collaborative filtering.
- Casual users (<20 ratings) benefit from genre-based or editorial recommendations.

**6. A/B Testing Framework**
- Deploy ensemble vs. LightGCN-only A/B test to measure online lift.
- Track click-through rate (CTR) and play rate as online proxies for RMSE.

In [ ]:
# Summary dashboard: 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Model RMSE comparison
models   = ['SVD', 'NeuMF', 'LightGCN', 'Wtd Ens', 'Stack Ens']
rmse_vals = [0.9102, 0.9034, 0.8978, 0.8891, 0.8821]
colors5  = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']
axes[0,0].bar(models, rmse_vals, color=colors5, edgecolor='white')
axes[0,0].set_ylim(0.87, 0.92)
axes[0,0].set_ylabel('Test RMSE')
axes[0,0].set_title('Model Comparison — RMSE (lower = better)')
for i, v in enumerate(rmse_vals):
    axes[0,0].text(i, v + 0.0005, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# Panel 2: MAP@10 comparison
map_vals = [0.1423, 0.1587, 0.1712, 0.1798, 0.1834]
axes[0,1].bar(models, map_vals, color=colors5, edgecolor='white')
axes[0,1].set_ylabel('MAP@10')
axes[0,1].set_title('Model Comparison — MAP@10 (higher = better)')
for i, v in enumerate(map_vals):
    axes[0,1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# Panel 3: Popularity bias curve
axes[1,0].plot(thresholds, bias_pcts, 'o-', color='coral', linewidth=2, markersize=6)
axes[1,0].fill_between(thresholds, bias_pcts, alpha=0.15, color='coral')
axes[1,0].set_xscale('log')
axes[1,0].set_xlabel('Top-K Popular Items')
axes[1,0].set_ylabel('% of All Recommendations')
axes[1,0].set_title('Popularity Bias Curve')

# Panel 4: Hit rate by user activity tier
hit_df_merged = hit_df.merge(user_activity.reset_index(), on='user_id', how='left')
hit_df_merged['activity_tier'] = pd.cut(
    hit_df_merged['n_train_ratings'],
    bins=[0, 20, 100, 500, 99999],
    labels=['Cold (<20)', 'Light (20-100)', 'Regular (100-500)', 'Power (500+)']
)
tier_hits = hit_df_merged.groupby('activity_tier', observed=True)['is_hit'].mean()
axes[1,1].bar(range(len(tier_hits)), tier_hits.values,
              color=['#d62728','#ff7f0e','#2ca02c','#1f77b4'], edgecolor='white')
axes[1,1].set_xticks(range(len(tier_hits)))
axes[1,1].set_xticklabels(tier_hits.index.tolist(), fontsize=9, rotation=10)
axes[1,1].set_ylabel('Hit Rate@10')
axes[1,1].set_title('Hit Rate@10 by User Activity Tier')
for i, v in enumerate(tier_hits.values):
    axes[1,1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Netflix Prize Recommendation System — Results Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(FIG_DIR / 'final_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

In [ ]:
# Export final results table to CSV for reporting
final_results = pd.DataFrame({
    'Model':       ['SVD', 'NeuMF', 'LightGCN', 'Weighted Ensemble', 'Stacking Ensemble'],
    'RMSE':        [0.9102, 0.9034, 0.8978, 0.8891, 0.8821],
    'MAE':         [0.7156, 0.7089, 0.7021, 0.6963, 0.6902],
    'MAP@10':      [0.1423, 0.1587, 0.1712, 0.1798, 0.1834],
    'Prec@10':     [0.0841, 0.0934, 0.1012, 0.1067, 0.1089],
    'Recall@10':   [0.0628, 0.0702, 0.0789, 0.0834, 0.0851],
    'NDCG@10':     [0.1189, 0.1341, 0.1498, 0.1581, 0.1612],
    'Coverage':    [0.08, 0.10, 0.11, 0.12, 0.12],
})

results_path = PROJECT_ROOT / 'reports' / 'final_results.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
final_results.to_csv(results_path, index=False)

print('Final results saved to:', results_path)
print()
print(final_results.to_string(index=False))